## Stage 5 - Evaluation & Verification 

In [5]:
"""
Steps 7-8: Evaluation & Verification
"""

import json
import logging
import pandas as pd
import numpy as np
from datetime import datetime
from difflib import SequenceMatcher
from typing import List, Dict

from config import (
    CHUNKS_PATH, BENCHMARK_PATH, RESULTS_PATH, 
    BENCHMARK_SAMPLES, RANDOM_SEED
)
from rag_core import RAGAgent

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class Evaluator:
    """Evaluation and benchmarking system."""
    
    def __init__(self):
        self.agent = RAGAgent()
    
    def create_benchmark(self, output_path: str = BENCHMARK_PATH):
        """
        Create benchmark dataset from papers.
        """
        logger.info("Creating benchmark dataset...")
        
        # Load papers
        records = []
        with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
            for line in f:
                records.append(json.loads(line))
        
        chunks_df = pd.DataFrame(records)
        
        # Get unique papers
        papers_df = chunks_df.drop_duplicates(subset=["doc_id"]).copy()
        
        # Sample papers
        sample = papers_df.sample(min(BENCHMARK_SAMPLES, len(papers_df)), 
                                  random_state=RANDOM_SEED)
        
        # Create questions
        benchmark_data = []
        for _, row in sample.iterrows():
            # Generate factual question about the paper
            question = f"What problem does the paper '{row['title']}' address?"
            
            # Expected answer (text)
            expected = row['text'][:500]  # Truncate for practicality
            
            # Append '...' to markdown
            benchmark_data.append({
                "question": question,
                "expected_answer": expected,
                "source_doc_id": row["doc_id"],
                "source_title": row["title"],
                "question_type": "factual",
                "created_at": datetime.now().isoformat()
            })
        
        benchmark_df = pd.DataFrame(benchmark_data)
        benchmark_df.to_csv(output_path, index=False)
        logger.info(f"Created {len(benchmark_df)} benchmark questions -> {output_path}")
        
        return benchmark_df
    
    def calculate_similarity(self, text1: str, text2: str) -> float:
        """Calculate text similarity."""
        return SequenceMatcher(None, text1.lower(), text2.lower()).ratio()
    
    def evaluate_single(self, question: str, expected: str) -> Dict:
        """
        Evaluate single question.
        """
        result = self.agent.query(question)
        
        # Calculate similarity with expected
        similarity = self.calculate_similarity(expected, result['answer'])
        
        # Enhanced verification
        verification = self._verify_answer_detailed(
            result['answer'], 
            result['sources'], 
            result['status']
        )
        
        return {
            "question": question,
            "expected_answer": expected[:100] + "..." if len(expected) > 100 else expected,
            "generated_answer": result.get['answer'],
            "status": result.get['status'],
            "confidence": result.get['confidence'],
            "verification": verification,
            "similarity_score": round(similarity, 3),
            "num_citations": len(result.get['sources']),
            "response_time": result.get['total_time'],
            "retrieval_time": result.get['retrieval_time'],
            "generation_time": result.get['generation_time']
        }
    
    def _verify_answer_detailed(self, answer: str, sources: List[Dict], 
                                base_status: str) -> str:
        """
        IMPROVED: Detailed verification logic.
        """
        if not sources:
            return "No Sources Retrieved"
        
        if not answer or len(answer) < 30:
            return "Empty or Too Short"
        
        if "not enough information" in answer.lower():
            return "Acknowledged Missing Info"
        
        # Check if answer uses citations
        has_citations = any(str(i) in answer for i in range(1, 10))
        
        if not has_citations:
            return "No Citations - Possibly Hallucinated"
        
        # Check citation count vs sources
        citation_count = sum(str(i) in answer for i in range(1, len(sources) + 1))
        
        if citation_count >= 2 and base_status == "Supported":
            return "Well Supported with Citations"
        
        return base_status
    
    def run_evaluation(self, benchmark_path: str = BENCHMARK_PATH):
        """
        Run full evaluation pipeline.
        """
        logger.info("=" * 60)
        logger.info("STEP 7: Running Evaluation")
        logger.info("=" * 60)
        
        # Initialize agent
        self.agent.initialize()
        
        # Load benchmark
        benchmark_df = pd.read_csv(benchmark_path)
        logger.info(f"Loaded {len(benchmark_df)} benchmark questions")
        
        # Evaluate each
        results = []
        for i, row in benchmark_df.iterrows():
            logger.info(f"Evaluating {i+1}/{len(benchmark_df)}: {row['question'][:50]}...")
            
            result = self.evaluate_single(row['question'], row['expected_answer'])
            results.append(result)
        
        # Create results DataFrame
        results_df = pd.DataFrame(results)
        
        # Add manual correctness column for future labeling
        results_df['manual_correctness'] = ''
        results_df['notes'] = ''
        
        # Save results
        results_df.to_csv(RESULTS_PATH, index=False)
        logger.info(f"\nResults saved to {RESULTS_PATH}")
        
        # Print summary
        self._print_summary(results_df)
        
        return results_df
    
    def _print_summary(self, results_df: pd.DataFrame):
        """Print evaluation summary."""
        print("\n" + "=" * 60)
        print("EVALUATION SUMMARY")
        print("=" * 60)
        
        print(f"\nTotal Questions: {len(results_df)}")
        print(f"Average Response Time: {results_df['response_time'].mean():.2f}s")
        print(f"Average Confidence: {results_df['confidence'].mean():.3f}")
        
        print("\nStatus Distribution:")
        print(results_df['status'].value_counts())
        
        print("\nVerification Distribution:")
        print(results_df['verification'].value_counts())
        
        print("\nSimilarity Scores:")
        print(f"  Mean: {results_df['similarity_score'].mean():.3f}")
        print(f"  Median: {results_df['similarity_score'].median():.3f}")
    
    def analyze_failures(self, results_df: pd.DataFrame = None):
        """
        STEP 8: Failure Analysis
        """
        if results_df is None:
            results_df = pd.read_csv(RESULTS_PATH)
        
        logger.info("=" * 60)
        logger.info("STEP 8: Failure Analysis")
        logger.info("=" * 60)
        
        # Identify failures
        failures = results_df[
            (results_df['status'] != 'Supported') |
            (results_df['verification'].str.contains('Hallucinated|No Sources', na=False))
        ]
        
        logger.info(f"\nIdentified {len(failures)} potential failures:")
        
        for _, row in failures.head(5).iterrows():
            print(f"\n--- Failure Case ---")
            print(f"Question: {row['question'][:60]}...")
            print(f"Status: {row['status']}")
            print(f"Verification: {row['verification']}")
            print(f"Confidence: {row['confidence']}")
        
        # Suggest improvements
        print("\n" + "=" * 60)
        print("IMPROVEMENT SUGGESTIONS")
        print("=" * 60)
        
        low_conf = results_df[results_df['confidence'] < 0.5]
        if len(low_conf) > 0:
            print(f"- {len(low_conf)} answers had low confidence. Consider increasing top_k.")
        
        no_cite = results_df[results_df['verification'].str.contains('No Citations', na=False)]
        if len(no_cite) > 0:
            print(f"- {len(no_cite)} answers lacked citations. Check prompt engineering.")


def run_full_evaluation():
    """Execute Steps 7-8."""
    evaluator = Evaluator()
    
    # Create benchmark if doesn't exist
    try:
        benchmark = pd.read_csv(BENCHMARK_PATH)
        logger.info(f"Loaded existing benchmark with {len(benchmark)} questions")
    except FileNotFoundError:
        evaluator.create_benchmark()
    
    # Run evaluation
    results = evaluator.run_evaluation()
    
    # Analyze failures
    evaluator.analyze_failures(results)
    
    logger.info("\n✅ Evaluation complete!")


if __name__ == "__main__":
    run_full_evaluation()

INFO:__main__:Loaded existing benchmark with 20 questions
INFO:__main__:============================================================
INFO:__main__:STEP 7: Running Evaluation
INFO:__main__:============================================================
INFO:rag_core:Initializing RAG Agent...
INFO:sentence_transformers.base.model:No device provided, using mps
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f2

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentenc

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

: 